In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters.xlsx")

# ─────────────────────────────────────────────────────────────
# 지표 계산: RSI, OBV, Bollinger Bands, MACD
# ─────────────────────────────────────────────────────────────
def compute_indicators(df):
    df = df.copy()
    # Bollinger Bands (20일 SMA ±2σ)
    df['BB_MID']   = df['종가'].rolling(20).mean()
    df['BB_STD']   = df['종가'].rolling(20).std()
    df['BB_UPPER'] = df['BB_MID'] + 2 * df['BB_STD']
    df['BB_LOWER'] = df['BB_MID'] - 2 * df['BB_STD']
    # MACD (12,26,9)
    ema12 = df['종가'].ewm(span=12, adjust=False).mean()
    ema26 = df['종가'].ewm(span=26, adjust=False).mean()
    df['MACD']     = ema12 - ema26
    df['MACD_SIG'] = df['MACD'].ewm(span=9, adjust=False).mean()
    # RSI (14) & Signal (9)
    delta = df['종가'].diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_g = gain.rolling(14).mean()
    avg_l = loss.rolling(14).mean()
    rs    = avg_g / avg_l
    df['RSI_14']   = 100 - (100 / (1 + rs))
    df['RSI_SIG9'] = df['RSI_14'].rolling(9).mean()
    # OBV & Signal (9)
    df['OBV']      = (np.sign(df['종가'].diff()) * df['거래량']).fillna(0).cumsum()
    df['OBV_SIG9'] = df['OBV'].rolling(9).mean()
    return df.dropna()

# ─────────────────────────────────────────────────────────────
# 트레이드 카운트 함수
# ─────────────────────────────────────────────────────────────
def count_trades(p, df_raw):
    cash, shares = 10000.0, 0.0
    buy_count = sell_count = 0
    df = compute_indicators(df_raw)
    for _, r in df.iterrows():
        price = r['종가']
        # 매수
        if shares == 0 and (
            r['OBV'] > r['OBV_SIG9'] and
            r['RSI_14'] < r['RSI_SIG9'] and
            r['종가'] < r['BB_LOWER'] * (1 + p['boll_buffer']) and
            r['MACD'] > r['MACD_SIG']
        ):
            shares, cash = cash / price, 0.0
            buy_count += 1
        # 매도
        elif shares > 0 and (
            (r['RSI_14'] > p['rsi_sell_th'] and
             r['OBV'] < r['OBV_SIG9'] and
             r['종가'] > r['BB_UPPER'] * (1 + p['boll_buffer']))
            or (r['MACD'] < r['MACD_SIG'])
        ):
            cash, shares = shares * price, 0.0
            sell_count += 1
    return buy_count, sell_count

# ─────────────────────────────────────────────────────────────
# ROI 계산
# ─────────────────────────────────────────────────────────────
def backtest_roi(p, df_raw):
    cash, shares = 10000.0, 0.0
    df = compute_indicators(df_raw)
    for _, r in df.iterrows():
        price = r['종가']
        # 매수
        if shares == 0 and (
            r['OBV'] > r['OBV_SIG9'] and
            r['RSI_14'] < r['RSI_SIG9'] and
            r['종가'] < r['BB_LOWER'] * (1 + p['boll_buffer']) and
            r['MACD'] > r['MACD_SIG']
        ):
            shares, cash = cash / price, 0.0
        # 매도
        elif shares > 0 and (
            (r['RSI_14'] > p['rsi_sell_th'] and
             r['OBV'] < r['OBV_SIG9'] and
             r['종가'] > r['BB_UPPER'] * (1 + p['boll_buffer']))
            or (r['MACD'] < r['MACD_SIG'])
        ):
            cash, shares = shares * price, 0.0
    final = cash + shares * df.iloc[-1]['종가']
    return (final - 10000.0) / 10000.0 * 100

# ─────────────────────────────────────────────────────────────
# 최적화 메인
# ─────────────────────────────────────────────────────────────
new_records = []
# 종목 선택
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})
print("🔔 처리 종목 선택:")
for i, comp in enumerate(available, 1): print(f" {i}. {comp}")
sel = input("인덱스 또는 all: ").strip().lower()
TARGET = available if sel=='all' else [available[int(x)-1] for x in sel.split(',')]
# 기간 입력
start_def, end_def = "2022-01-01","2025-06-24"
start = input(f"시작 YYYY-MM-DD (기본 {start_def}): ") or start_def
end   = input(f"종료 YYYY-MM-DD (기본 {end_def}): ") or end_def
for comp in TARGET:
    path = glob.glob(os.path.join(PROCESSED_FOLDER, f"{comp}_*_지표포함.csv"))[0]
    df_raw = pd.read_csv(path, encoding='utf-8-sig')
    df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
    df_raw = df_raw[(df_raw['날짜']>=start)&(df_raw['날짜']<=end)]
    print(f"🔍 {comp} 최적화")
    # TPE
    def obj_tpe(tr):
        return backtest_roi({
            'rsi_sell_th': tr.suggest_float('rsi_sell_th',30,100),
            'boll_buffer': tr.suggest_float('boll_buffer',0,0.1)
        }, df_raw)
    study = optuna.create_study(direction='maximize')
    study.optimize(obj_tpe,n_trials=500)
    best = study.best_params
    imp   = get_param_importances(study)
    imp_k = [k for k,v in imp.items() if v>0.05]
    # CMA
    bounds={'rsi_sell_th':(30,100),'boll_buffer':(0,0.1)}
    narrow={k:(max(bounds[k][0],best[k]-0.2*(bounds[k][1]-bounds[k][0])),min(bounds[k][1],best[k]+0.2*(bounds[k][1]-bounds[k][0]))) for k in imp_k}
    def obj_cma(tr):
        p=best.copy()
        for k in imp_k: p[k]=tr.suggest_float(k,*narrow[k])
        return backtest_roi(p,df_raw)
    cma=optuna.create_study(direction='maximize',sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma,n_trials=200)
    best.update(cma.best_params)
    roi=cma.best_value
    buys,sells = count_trades(best, df_raw)
    rec={'종목':comp,'Start':start,'End':end,'ROI(%)':round(roi,2),'buys':buys,'sells':sells}
    rec.update(best); rec.update({f'imp_{k}':imp.get(k,0) for k in imp})
    new_records.append(rec)
    print(f"✅ {comp} ROI={roi:.2f}% buys={buys} sells={sells}")
# 결과 저장
if os.path.exists(PARAMETERS_PATH):
    df_exist = pd.read_excel(PARAMETERS_PATH)
    df_upd   = pd.concat([df_exist,pd.DataFrame(new_records)],ignore_index=True)
else:
    df_upd = pd.DataFrame(new_records)
if 'Index' in df_upd.columns: df_upd.drop(columns=['Index'],inplace=True)
df_upd.insert(0,'Index',range(1,len(df_upd)+1))
df_upd.to_excel(PARAMETERS_PATH,index=False)
print(f"Parameters updated → {PARAMETERS_PATH}")


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정 (optimize_all.py와 동일)
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
RESULTS_ROOT     = r"C:\Users\LabPC\OneDrive\주식\Results"
PARAM_FILE       = os.path.join(RESULTS_ROOT, "Parameters", "parameters.xlsx")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 지표 계산 함수 (optimize_all.py와 동일)
# ─────────────────────────────────────────────────────────────
def compute_indicators(df):
    df = df.copy()
    df['BB_MID']   = df['종가'].rolling(20).mean()
    df['BB_STD']   = df['종가'].rolling(20).std()
    df['BB_UPPER'] = df['BB_MID'] + 2 * df['BB_STD']
    df['BB_LOWER'] = df['BB_MID'] - 2 * df['BB_STD']
    ema12 = df['종가'].ewm(span=12, adjust=False).mean()
    ema26 = df['종가'].ewm(span=26, adjust=False).mean()
    df['MACD']     = ema12 - ema26
    df['MACD_SIG'] = df['MACD'].ewm(span=9, adjust=False).mean()
    delta = df['종가'].diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_g = gain.rolling(14).mean()
    avg_l = loss.rolling(14).mean()
    rs    = avg_g / avg_l
    df['RSI_14']   = 100 - (100 / (1 + rs))
    df['RSI_SIG9'] = df['RSI_14'].rolling(9).mean()
    df['OBV']      = (np.sign(df['종가'].diff()) * df['거래량']).fillna(0).cumsum()
    df['OBV_SIG9'] = df['OBV'].rolling(9).mean()
    return df.dropna()

# ─────────────────────────────────────────────────────────────
# 매수/매도 신호 함수 (optimize_all.py와 동일)
# ─────────────────────────────────────────────────────────────
def is_buy(r, p):
    return (
        r['OBV']      > r['OBV_SIG9'] and
        r['RSI_14']   < r['RSI_SIG9'] and
        r['종가']     < r['BB_LOWER'] * (1 + p['boll_buffer']) and
        r['MACD']     > r['MACD_SIG']
    )

def is_sell(r, p):
    cond1 = (
        r['RSI_14'] > p['rsi_sell_th'] and
        r['OBV']    < r['OBV_SIG9'] and
        r['종가']   > r['BB_UPPER'] * (1 + p['boll_buffer'])
    )
    cond2 = (r['MACD'] < r['MACD_SIG'])
    return cond1 or cond2

# ─────────────────────────────────────────────────────────────
# 백테스트 실행 함수 (optimize_all.py와 동일)
# ─────────────────────────────────────────────────────────────
def run_backtest(df, params, extra_on_buy=False, cooldown_days=0):
    cash, shares = 10000.0, 0.0
    total_injected = 10000.0
    logs = []
    last_buy_date = None
    df_ind = compute_indicators(df)

    for _, r in df_ind.iterrows():
        date, price = r['날짜'], r['종가']
        ok = last_buy_date is None or (date - last_buy_date).days >= cooldown_days
        # 매수
        if shares == 0 and is_buy(r, params) and ok:
            if extra_on_buy:
                cash += 10000.0
                total_injected += 10000.0
            shares, cash = cash/price, 0.0
            last_buy_date = date
            logs.append([date, 'BUY', price, shares, cash, shares*price, total_injected])
        # 매도
        elif shares > 0 and is_sell(r, params):
            cash, shares = shares*price, 0.0
            last_buy_date = None
            logs.append([date, 'SELL', price, shares, cash, cash, total_injected])
    # 최종 청산
    if shares > 0:
        date, price = df_ind.iloc[-1]['날짜'], df_ind.iloc[-1]['종가']
        cash += shares*price
        shares = 0.0
        logs.append([date, 'LIQUIDATE', price, shares, cash, cash, total_injected])

    cols = ['날짜','액션','가격','보유주','현금','총자산','투입금액']
    out = pd.DataFrame(logs, columns=cols)
    if not out.empty:
        out['ROI(%)'] = (out['총자산']/out['투입금액']*100).round(2).astype(str)+'%'
    return out

# ─────────────────────────────────────────────────────────────
# 1) parameters 불러오기 및 선택
# ─────────────────────────────────────────────────────────────
dfp = pd.read_excel(PARAM_FILE)
print("🔔 시뮬레이션 가능 인덱스 목록:")
for _, row in dfp.iterrows():
    print(f" {int(row.Index)}. {row.종목} ({row.Start}~{row.End}, ROI:{row['ROI(%)']}%)")
sel = input("\n시뮬레이션할 Index(콤마 or all): ").strip().lower()
sel_idx = dfp if sel=='all' else dfp[dfp.Index.isin([int(x) for x in sel.split(',')])]

# ▶ 날짜 지정
custom_start, custom_end = [], []
for _, r in sel_idx.iterrows():
    print(f"\n📌 종목: {r.종목}")
    s = input(" 시작일 (YYYY-MM-DD): ")
    e = input(" 종료일 (YYYY-MM-DD): ")
    custom_start.append(pd.to_datetime(s))
    custom_end.append(pd.to_datetime(e))
sel_idx['Start'] = custom_start
sel_idx['End']   = custom_end

print("\n▶ 최종 시뮬레이션 목록:")
print(sel_idx[['Index','종목','Start','End','ROI(%)']].to_string(index=False))

# ─────────────────────────────────────────────────────────────
# 2) 각 시뮬레이션 실행 및 결과 저장
# ─────────────────────────────────────────────────────────────
for _, r in sel_idx.iterrows():
    idx, comp = int(r.Index), r.종목
    start, end = r.Start, r.End
    params = {k: r[k] for k in ['rsi_sell_th','boll_buffer']}
    # 원본 데이터 로드
    fp = glob.glob(os.path.join(PROCESSED_FOLDER, f"{comp}_*_지표포함.csv"))[0]
    df_raw = pd.read_csv(fp, encoding='utf-8-sig')
    df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
    df = df_raw[(df_raw['날짜']>=start)&(df_raw['날짜']<=end)].reset_index(drop=True)
    # ① 한번만 투자
    df_once = run_backtest(df, params, extra_on_buy=False)
    if not df_once.empty:
        fname = f"{idx}_{comp}_once_{start.date()}_{end.date()}_ROI_{df_once['ROI(%)'].iloc[-1]}.csv"
        df_once.to_csv(os.path.join(RESULTS_ROOT, comp, fname), index=False, encoding='utf-8-sig')
    # ② 매수마다 추가투자
    df_extra= run_backtest(df, params, extra_on_buy=True)
    if not df_extra.empty:
        fname= f"{idx}_{comp}_extra_{start.date()}_{end.date()}_ROI_{df_extra['ROI(%)'].iloc[-1]}.csv"
        df_extra.to_csv(os.path.join(RESULTS_ROOT, comp, fname), index=False, encoding='utf-8-sig')
    # Baseline
    print(f"✅ {comp} 시뮬레이션 완료: once_trades={len(df_once)}, extra_trades={len(df_extra)}")
